# EDA/Preprocessing

In [3]:
# Ensure that data is as described
import pandas as pd

test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Ecs172/test.csv")
train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Ecs172/train.csv")
items = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Ecs172/item_metadata.csv")

print("Shape of training data:", train.shape)
print("Shape of testing data", test.shape)
print("Shape of item metadata", items.shape)

Shape of training data: (32206, 5)
Shape of testing data (10000, 2)
Shape of item metadata (8681, 9)


In [4]:
#check for duplicates and any missing values
print("Duplicates:", train.duplicated().sum())
print("NULL:", train.isnull().sum(), items.isnull().sum())
print("Duplicates:", items.duplicated().sum())
print("NULL:", items.isnull().sum())

Duplicates: 0
NULL: user_id        0
item_id        0
rating         0
review_text    0
timestamp      0
dtype: int64 item_id              0
title                0
description         42
features            37
categories        8240
main_category        6
store                0
price              592
average_rating       0
dtype: int64
Duplicates: 0
NULL: item_id              0
title                0
description         42
features            37
categories        8240
main_category        6
store                0
price              592
average_rating       0
dtype: int64


In [5]:
#Look at the rating counts
train.describe()

,rating,timestamp
count,32206.000000,3.220600e+04
mean,3.986121,1.469506e+12
std,1.357766,8.270879e+10
min,1.000000,1.073708e+12
25%,3.000000,1.408207e+12
50%,5.000000,1.455749e+12
75%,5.000000,1.529269e+12
max,5.000000,1.681262e+12


Our dataset has a very positive bias as it is skewed very heavily towards 5 stars. Considering that the median and the 75th percentile are both 5.0, this must mean that over half of our data consists of solely 5 star reviews. Taking a closer look below we see that over 17,196 of the 32206 ratings are actually 5 stars.

In [6]:
# specific count of ratings
print(train['rating'].value_counts())

rating
5.0    17196
4.0     6138
3.0     3599
1.0     3498
2.0     1775
Name: count, dtype: int64


Considering the nature of the problem, the best course of action for the train/test split, is to try to predict an user's most recent review based on the other 4 products that they have bought.

In [7]:
# Grab the most recent purchase by every user, the rest will go into training for user profiles
train = train.sort_values(['user_id', 'timestamp'])
test_split = train.groupby('user_id').tail(1)
train_split = train.drop(test_split.index)

print(f"Train Split: {len(train_split)}")
print(f"Test split: {len(test_split)}")

Train Split: 30206
Test split: 2000


In [8]:
#vectorize, I had help with AI in this part of the code, I was not too sure how to vectorize everything so that I could implement TF-IDF
items['content_soup'] = (
    items['title'].astype(str).fillna('') + " " +
    items['description'].astype(str).fillna('') + " " +
    items['features'].astype(str).fillna('') + " " +
    items['categories'].astype(str).fillna('') + " " +
    items['main_category'].astype(str).fillna('')
)

#lowercase everything
items['content_soup'] = items['content_soup'].str.lower()

# Implementation


In [28]:
#Start with creating a baseline model, 5000 was a recommended fit
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(items['content_soup'])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (8681, 5000)


In [48]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

mapping = {item_id: i for i, item_id in enumerate(items['item_id'])}
def get_user_profile(id, interaction_df, tfidf_matrix, item_id_to_idx, w):
    # filter training data for specific user history
    user_data = interaction_df[interaction_df['user_id'] == id]
    user_item_ids = user_data['item_id'].values
    user_ratings = user_data['rating'].values

    indices = [item_id_to_idx[iid] for iid in user_item_ids if iid in item_id_to_idx]

    if not indices:
        return None

    item_vectors = tfidf_matrix[indices]

    # ensure that higher ratings are weighted more heavily
    weights = np.array(user_ratings**w).reshape(-1, 1)

    # use the weighted average to generate user profiles
    weighted_vectors = item_vectors.multiply(weights)
    user_profile = np.asarray(weighted_vectors.sum(axis=0) / weights.sum())

    return user_profile

def evaluate_accuracy_tuned(val_df, train_df, tfidf_matrix, item_id_to_idx, w):
    hits = 0
    total_users = len(val_df)
    all_item_ids = list(item_id_to_idx.keys())

    for _, row in val_df.iterrows():
        id = row['user_id']
        true_item = row['item_id']

        # Call our tuned profile function
        profile = get_user_profile(id, train_df, tfidf_matrix, item_id_to_idx, w)

        if profile is None: continue

        # Create 4 decoys + set aside test case
        other_items = [i for i in all_item_ids if i != true_item]
        decoys = np.random.choice(other_items, 4, replace=False)
        candidates = [true_item] + list(decoys)

        candidate_scores = []
        for cand_id in candidates:
            cand_vec = tfidf_matrix[item_id_to_idx[cand_id]]
            # Measure similarity using cosine similarity
            score = cosine_similarity(profile, cand_vec)[0][0]
            candidate_scores.append((cand_id, score))

        # Rank the scores
        candidate_scores.sort(key=lambda x: x[1], reverse=True)
        if candidate_scores[0][0] == true_item:
            hits += 1

    return (hits / total_users) * 100

final_result = evaluate_accuracy_tuned(test_split, train_split, tfidf_matrix, mapping, w=1)
print(f"Base model hit rate: {final_result}%")

Base model hit rate: 59.550000000000004%


In [41]:
def hyperparameter_tuning(max_features, rating_weight_power):
    tfidf = TfidfVectorizer(stop_words='english', max_features=max_features)
    matrix = tfidf.fit_transform(items['content_soup'])
    mapping = {item_id: i for i, item_id in enumerate(items['item_id'])}

    # Run evaluation with the current power weight
    accuracy = evaluate_accuracy_tuned(test_split, train_split, matrix, mapping, rating_weight_power)

    return accuracy

# I manually sorted through the options
feature_options = [5500]
power_options = [1,2]

results_grid = []
for f in feature_options:
    for p in power_options:
        current_acc = hyperparameter_tuning(f, p)

        results_grid.append({
            'Features': f,
            'weight power': p,
            'Hit Rate': current_acc
        })

results = pd.DataFrame(results_grid)
results= results_df.sort_values(by='Hit Rate @ 1', ascending=False)
print(results.to_string(index=False))

best = results.iloc[0]

 Max Features  Rating Power  Hit Rate @ 1
         5500             2          60.4
         5500             1          58.8


# Final Model

In [47]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5500)
tfidf_matrix = tfidf.fit_transform(items['content_soup'])

mapping = {item_id: i for i, item_id in enumerate(items['item_id'])}

def get_user_profile(user_id, interaction_df, tfidf_matrix, item_id_to_idx, w=2):
    user_data = interaction_df[interaction_df['user_id'] == user_id]
    user_item_ids = user_data['item_id'].values
    user_ratings = user_data['rating'].values

    indices = [item_id_to_idx[iid] for iid in user_item_ids if iid in item_id_to_idx]

    if not indices:
        return None

    vectors = tfidf_matrix[indices]
    weights = np.array(user_ratings**w).reshape(-1, 1)

    # Weighted average of item profiles
    weighted_vectors = vectors.multiply(weights)
    user_profile = np.asarray(weighted_vectors.sum(axis=0) / weights.sum())

    return user_profile

results = []

# Group the test data by user to rank their 5 specific candidates
for uid, group in test.groupby('user_id'):
    candidate_ids = group['item_id'].tolist()

    # Build profile using all training data
    profile = get_user_profile(uid, train, tfidf_matrix, mapping, w=2)

    #rank items from the test set
    scores = []
    for cid in candidate_ids:
        if cid in mapping:
            cand_vec = tfidf_matrix[mapping[cid]]
            score = cosine_similarity(profile, cand_vec)[0][0]
            scores.append((cid, score))

    # sort by cosine similarity score and append to final list
    scores.sort(key=lambda x: x[1], reverse=True)
    sorted_ids = [s[0] for s in scores]
    results.append([uid] + sorted_ids)

cols = ['user_id', 'rank_1', 'rank_2', 'rank_3', 'rank_4', 'rank_5']
final_results = pd.DataFrame(results, columns=cols)

final_results.to_csv('submission.csv', index=False)